# Lightweight HBCC — CIFAR-10 Full Training Pipeline

This notebook orchestrates the full workflow implemented in this repository:

1. Environment and smoke checks
2. Teacher/reference training
3. CoC baseline training
4. Student CE-only training
5. Short proxy architecture search
6. Technique ablations
7. Knowledge distillation
8. Structured pruning export and fine-tune
9. Benchmark matrix and Pareto summary
10. Published results table

The default flags are conservative. Turn on the phases you want to run before executing all cells.

CIFAR-10 training uses `train=45k` and `val=5k` from the original training split. The official CIFAR-10 test split is evaluated once on `best.pth` and reported as `test_acc1`. See `context-cluster-cifar100.ipynb` for the matching CIFAR-100 Kaggle pipeline.


In [1]:
!git clone https://github.com/nvhungus/HBCC.git
%cd HBCC


Cloning into 'Lightweight-Context-Cluster'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 143 (delta 73), reused 124 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 7.20 MiB | 27.09 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/kaggle/working/Lightweight-Context-Cluster


In [2]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Run this notebook from the repo root or notebooks dir. Got {ROOT}"

COC_PYTHON = Path(r"D:\Anaconda\envs\CoC\python.exe")
PYTHON = COC_PYTHON if COC_PYTHON.exists() else Path(sys.executable)

print("Repo:", ROOT)
print("Python:", PYTHON)

Repo: /kaggle/working/Lightweight-Context-Cluster
Python: /usr/bin/python3


## Phase Switches

Set the flags below. For a first run, keep only `RUN_SMOKE = True`. For the full research run, enable each phase deliberately.

In [3]:
RUN_ENV_CHECKS = True
RUN_SMOKE = False

RUN_TEACHER = True
RUN_LIGHTWEIGHT_BASELINES = True
RUN_COC_BASELINE = True
RUN_STUDENT_CE = True
RUN_PROXY_SEARCH = False
RUN_ABLATIONS = False
RUN_KD = False
RUN_PRUNING = True
RUN_BENCHMARKS = True
RUN_PARETO_REPORT = True

FULL_EPOCHS = 200
ACCURACY_EPOCHS = 300
PROXY_EPOCHS = 30
PRUNING_EPOCHS = 80
PRUNED_FINETUNE_EPOCHS = 80

COMMON_TRAIN_OVERRIDES = [
    # Reduce these if you hit OOM.
    # "data.batch_size=64",
    # "data.val_batch_size=128",
    # "data.test_batch_size=128",
]

BENCHMARK_BATCHES = [1, 16, 64, 128]
BENCHMARK_WARMUP = 30
BENCHMARK_RUNS = 100

# Use shorter benchmark settings while debugging the notebook.
DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP = 2
DEBUG_BENCHMARK_RUNS = 3

In [4]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    """Run a repo Python command; tqdm updates are kept to one notebook line."""
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def train(config: str, output: str, overrides: list[str] | None = None, extra: list[str] | None = None) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args([*(overrides or []), *COMMON_TRAIN_OVERRIDES])
    args += extra or []
    run_py(args)


def benchmark(config: str, checkpoint: str | None = None, debug: bool = False, profile: bool = True) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup = DEBUG_BENCHMARK_WARMUP if debug else BENCHMARK_WARMUP
    runs = DEBUG_BENCHMARK_RUNS if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py",
        "--config",
        config,
        "--output",
        "results/benchmark",
        "--batch-sizes",
        *[str(v) for v in batches],
        "--warmup",
        str(warmup),
        "--runs",
        str(runs),
    ]
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"Skip checkpoint for {config}; not found: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)

## 0. Environment and Smoke Checks

In [5]:
if RUN_ENV_CHECKS:
    run_py(["-m", "pytest"])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_latency_tiny.yaml", "--batch-size", "1"])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_current_reference.yaml", "--batch-size", "1"])

if RUN_SMOKE:
    train(
        "configs/smoke.yaml",
        "runs_smoke",
        extra=["--limit-train-batches", "1", "--limit-val-batches", "1", "--limit-test-batches", "1"],
    )
    benchmark("configs/smoke.yaml", debug=True, profile=False)


$ /usr/bin/python3 -m pytest


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working/Lightweight-Context-Cluster
configfile: pyproject.toml
testpaths: tests
plugins: langsmith-0.7.30, anyio-4.13.0, typeguard-4.5.1
collected 7 items

tests/test_config_and_pruning.py ..                                      [ 28%]
tests/test_data.py ..                                                    [ 57%]
tests/test_models.py ...                                                 [100%]

============================== 7 passed in 10.28s ==============================

[exit=0] elapsed=14.2s

$ /usr/bin/python3 tools/shape_trace.py --config configs/hbcc_latency_tiny.yaml --batch-size 1


stem                                             -> (1, 32, 16, 16)
stages.0                                         -> (1, 32, 16, 16)
downsamples.0                                    -> (1, 48, 8, 8)
stages.1.blocks.0.cluster                        -> (1, 24, 8, 8)
stages.1                                         -> (1, 48, 8, 8)
downsamples.1                                    -> (1, 96, 4, 4)
stages.2.blocks.0.cluster                        -> (1, 96, 4, 4)
stages.2.blocks.1.cluster                        -> (1, 96, 4, 4)
stages.2                                         -> (1, 96, 4, 4)
downsamples.2                                    -> (1, 160, 2, 2)
stages.3.blocks.0.cluster                        -> (1, 160, 2, 2)
stages.3                                         -> (1, 160, 2, 2)
logits                                           -> (1, 10)

[exit=0] elapsed=5.4s

$ /usr/bin/python3 tools/shape_trace.py --config configs/hbcc_current_reference.yaml --batch-size 1


stem                                             -> (1, 32, 32, 32)
stages.0.blocks.0.cluster                        -> (1, 32, 32, 32)
stages.0                                         -> (1, 32, 32, 32)
downsamples.0                                    -> (1, 64, 16, 16)
stages.1.blocks.0.cluster                        -> (1, 32, 16, 16)
stages.1                                         -> (1, 64, 16, 16)
downsamples.1                                    -> (1, 128, 8, 8)
stages.2.blocks.0.cluster                        -> (1, 128, 8, 8)
stages.2.blocks.1.cluster                        -> (1, 128, 8, 8)
stages.2                                         -> (1, 128, 8, 8)
downsamples.2                                    -> (1, 192, 4, 4)
stages.3.blocks.0.cluster                        -> (1, 192, 4, 4)
stages.3                                         -> (1, 192, 4, 4)
logits                                           -> (1, 10)

[exit=0] elapsed=4.8s


## 1. Teacher and Reference Baselines

Train ResNet-18 first. It is both the accuracy reference and the default KD teacher.

In [6]:
if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        "runs_teacher",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        "runs_baselines",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        "runs_baselines",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )


$ /usr/bin/python3 tools/train.py --config configs/baselines/resnet18_cifar.yaml --output runs_teacher --override train.epochs=200


```text
test:  68%|██████▊   | 27/40 [00:01<00:00, 26.56it/s]
```



                                                                                   

                                                    
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 1.57550105138382,
  "train_acc1": 50.22480413105413,
  "train_time_s": 16.837167969999882,
  "val_loss": 1.0965907651901246,
  "val_acc1": 61.15999998168945,
  "val_time_s": 0.838542426999993
}

                                                                                   

                                                    
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 1.2093236076186524,
  "train_acc1": 68.28481125356126,
  "train_time_s": 15.908664105000298,
  "val_loss": 0.9106163222312927,
  "val_acc1": 69.55999993896485,
  "val_time_s": 0.7886125560003165
}

                                                                                   

                                                    
{
  "epoch": 2,
  "lr": 0.0009994449374809851,
  "train_loss": 1.

```text
test: 100%|██████████| 40/40 [00:02<00:00, 12.38it/s]
```


                                                                                   

                                                    
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 1.9239942287108158,
  "train_acc1": 31.541577635327634,
  "train_time_s": 18.214691743999538,
  "val_loss": 1.6019420761108398,
  "val_acc1": 41.23999993286133,
  "val_time_s": 3.1111576330004027
}

                                                                                   

                                                    
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 1.626268497219792,
  "train_acc1": 47.6963141025641,
  "train_time_s": 13.518823012000212,
  "val_loss": 1.268910332107544,
  "val_acc1": 55.3,
  "val_time_s": 0.7818798430007519
}

                                                                                   

                                                    
{
  "epoch": 2,
  "lr": 0.0009994449374809851,
  "train_loss": 1.4741674862016

```text
test:  70%|███████   | 28/40 [00:01<00:00, 27.26it/s]
```


                                                                                   

                                                    
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 1.8712113871533647,
  "train_acc1": 34.991542022792025,
  "train_time_s": 18.18626550100089,
  "val_loss": 1.4988383785247803,
  "val_acc1": 45.01999995727539,
  "val_time_s": 2.392623558998821
}

                                                                                   

                                                    
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 1.587371565337874,
  "train_acc1": 49.30332977207977,
  "train_time_s": 15.167197950000627,
  "val_loss": 1.299779926109314,
  "val_acc1": 53.34000001831055,
  "val_time_s": 0.7861846089999744
}

                                                                                   

                                                    
{
  "epoch": 2,
  "lr": 0.0009994449374809851,
  "train_loss": 1.4

## 2. CoC Baseline

Train the CIFAR-adapted CoC baseline separately so benchmark records can use its trained checkpoint instead of an untrained model.

In [7]:
COC_BASELINE_RUN = "runs_coc_baseline/coc_cifar_baseline"
COC_BASELINE_CHECKPOINT = f"{COC_BASELINE_RUN}/best.pth"

if RUN_COC_BASELINE:
    train(
        "configs/coc_cifar_baseline.yaml",
        "runs_coc_baseline",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    run_py(["tools/summarize_training.py", COC_BASELINE_RUN])



$ /usr/bin/python3 tools/train.py --config configs/coc_cifar_baseline.yaml --output runs_coc_baseline --override train.epochs=200


```text
test: 100%|██████████| 40/40 [00:01<00:00, 26.09it/s]
```


                                                                                   

                                                    
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 1.772810147358821,
  "train_acc1": 40.16871438746439,
  "train_time_s": 21.41202966199853,
  "val_loss": 1.322885889816284,
  "val_acc1": 52.920000024414065,
  "val_time_s": 1.1115458919994126
}

                                                                                   

                                                   
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 1.5113427802028818,
  "train_acc1": 53.3676103988604,
  "train_time_s": 19.71562312200149,
  "val_loss": 1.1592070560455323,
  "val_acc1": 59.55999993286133,
  "val_time_s": 0.9851982049985963
}

                                                                                   

                                                   
{
  "epoch": 2,
  "lr": 0.0009994449374809851,
  "train_loss": 1.3894

Best val: epoch=180, val_acc1=89.24, val_loss=0.4511
Held-out test: checkpoint=best.pth, epoch=180, test_acc1=87.89, test_loss=0.4861

[exit=0] elapsed=0.1s


## 3. Student CE-only Training

This is the fair baseline for each HBCC student before applying KD or pruning.

In [8]:
if RUN_STUDENT_CE:
    train(
        "configs/hbcc_accuracy_small.yaml",
        "runs_accuracy",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        "runs_accuracy",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )


$ /usr/bin/python3 tools/train.py --config configs/hbcc_accuracy_small.yaml --output runs_accuracy --override train.epochs=300


```text
test:  92%|█████████▎| 37/40 [00:01<00:00, 24.45it/s]
```


                                                                                   

                                                    
{
  "epoch": 0,
  "lr": 0.000208,
  "train_loss": 2.245701107883725,
  "train_acc1": 15.121972934472934,
  "train_time_s": 29.057841707999614,
  "val_loss": 2.0019828443527223,
  "val_acc1": 29.239999990844726,
  "val_time_s": 1.5974703280007816
}

                                                                                   

                                                   
{
  "epoch": 1,
  "lr": 0.00040599999999999995,
  "train_loss": 2.0564422848557475,
  "train_acc1": 23.54211182336182,
  "train_time_s": 26.34826413800147,
  "val_loss": 1.5086335763931273,
  "val_acc1": 47.2199999633789,
  "val_time_s": 0.9286540619996231
}

                                                                                   

                                                    
{
  "epoch": 2,
  "lr": 0.0006039999999999999,
  "train_loss": 1.938968173119

```text
test:  82%|████████▎ | 33/40 [00:01<00:00, 21.34it/s]
```


                                                                                   

                                                    
{
  "epoch": 0,
  "lr": 0.000208,
  "train_loss": 2.227630878785397,
  "train_acc1": 16.582086894586894,
  "train_time_s": 29.4842134290011,
  "val_loss": 1.946656749534607,
  "val_acc1": 29.84000001220703,
  "val_time_s": 1.785729655999603
}

                                                                                   

                                                   
{
  "epoch": 1,
  "lr": 0.00040599999999999995,
  "train_loss": 2.033288608589064,
  "train_acc1": 24.43019943019943,
  "train_time_s": 29.578654417000507,
  "val_loss": 1.4961044242858887,
  "val_acc1": 47.19999999389648,
  "val_time_s": 1.0685808130001533
}

                                                                                   

                                                   
{
  "epoch": 2,
  "lr": 0.0006039999999999999,
  "train_loss": 1.9218465748675528,

## 4. Short Proxy Architecture Search

Use 20-50 epochs to reject weak configurations. Only top candidates should get full training.

In [9]:
PROXY_JOBS = [
    {
        "name": "proxy_dims24_depth1111_mlp2",
        "overrides": [
            "model.embed_dims=[24,48,96,160]",
            "model.depths=[1,1,1,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_depth1121_mlp2",
        "overrides": [
            "model.embed_dims=[32,48,96,160]",
            "model.depths=[1,1,2,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_64_depth1221_mlp3",
        "overrides": [
            "model.embed_dims=[32,64,128,192]",
            "model.depths=[1,2,2,1]",
            "model.mlp_ratios=3.0",
            "model.heads=[1,2,4,4]",
        ],
    },
]

if RUN_PROXY_SEARCH:
    for job in PROXY_JOBS:
        train(
            "configs/hbcc_latency_tiny.yaml",
            "runs_proxy",
            overrides=[
                f"experiment.name={job['name']}",
                f"train.epochs={PROXY_EPOCHS}",
                *job["overrides"],
            ],
        )

## 5. Independent Ablations

Run each technique independently and decide `keep`, `drop`, or `defer` from metrics.

In [10]:
ABLATION_CONFIGS = [
    "configs/ablations/hbcc_tiny_hamming_late.yaml",
    "configs/ablations/hbcc_tiny_lbpconv.yaml",
]

if RUN_ABLATIONS:
    for cfg in ABLATION_CONFIGS:
        train(cfg, "runs_ablations", overrides=[f"train.epochs={FULL_EPOCHS}"])

## 6. Knowledge Distillation

Requires a trained teacher checkpoint. The teacher is used only during training; inference latency of the student is unchanged.

In [11]:
TEACHER_CHECKPOINT = "runs_teacher/resnet18_cifar/best.pth"

if RUN_KD:
    if not (ROOT / TEACHER_CHECKPOINT).exists():
        raise FileNotFoundError(f"Train the teacher first: {TEACHER_CHECKPOINT}")
    train(
        "configs/hbcc_latency_tiny.yaml",
        "runs_kd",
        overrides=[
            f"train.epochs={FULL_EPOCHS}",
            "train.kd_alpha=0.5",
            "train.kd_temperature=4.0",
        ],
        extra=[
            "--teacher-config",
            "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint",
            TEACHER_CHECKPOINT,
        ],
    )

## 7. Structured Pruning Export

Do not judge latency from the soft-mask model. Export a materialized smaller config, fine-tune it, then benchmark the exported model.

In [12]:
PRUNING_BASE_CHECKPOINT = "runs_accuracy/hbcc_accuracy_small/best.pth"
PRUNING_CONFIG = "configs/ablations/hbcc_accuracy_small_pruning_mask.yaml"
PRUNING_RUN_DIR = "runs_pruning_accuracy"
PRUNING_EXPERIMENT = "hbcc_accuracy_small_pruning_mask"
PRUNING_CHECKPOINT = f"{PRUNING_RUN_DIR}/{PRUNING_EXPERIMENT}/best.pth"
PRUNING_THRESHOLD = "0.90"
PRUNED_CONFIG = "configs/generated_ablations/hbcc_accuracy_small_pruned_export.yaml"
PRUNED_FINETUNE_RUN_DIR = "runs_pruned_accuracy"
PRUNED_FINETUNE_CHECKPOINT = "runs_pruned_accuracy/hbcc_accuracy_small_pruned_export/best.pth"

if RUN_PRUNING:
    if not (ROOT / PRUNING_BASE_CHECKPOINT).exists():
        raise FileNotFoundError(f"Train hbcc_accuracy_small first: {PRUNING_BASE_CHECKPOINT}")
    train(
        PRUNING_CONFIG,
        PRUNING_RUN_DIR,
        overrides=[f"train.epochs={PRUNING_EPOCHS}"],
        extra=["--resume", PRUNING_BASE_CHECKPOINT],
    )
    run_py([
        "tools/inspect_pruning_masks.py",
        "--config",
        PRUNING_CONFIG,
        "--checkpoint",
        PRUNING_CHECKPOINT,
        "--thresholds",
        "0.5",
        "0.7",
        "0.8",
        "0.9",
        "0.95",
    ])
    run_py([
        "tools/export_pruned.py",
        "--config",
        PRUNING_CONFIG,
        "--checkpoint",
        PRUNING_CHECKPOINT,
        "--output",
        PRUNED_CONFIG,
        "--threshold",
        PRUNING_THRESHOLD,
        "--min-channels",
        "16",
        "--round-to",
        "8",
    ])
    train(
        PRUNED_CONFIG,
        PRUNED_FINETUNE_RUN_DIR,
        overrides=[f"train.epochs={PRUNED_FINETUNE_EPOCHS}"],
    )



$ /usr/bin/python3 tools/train.py --config configs/ablations/hbcc_accuracy_small_pruning_mask.yaml --output runs_pruning_accuracy --override train.epochs=80 --resume runs_accuracy/hbcc_accuracy_small/best.pth


```text
test:  90%|█████████ | 36/40 [00:01<00:00, 23.87it/s]
```


                                                                                   

                                                    
{
  "epoch": 0,
  "lr": 0.0001999229036240723,
  "train_loss": 0.6283230604948821,
  "train_acc1": 94.87624643874643,
  "train_time_s": 30.836233476002235,
  "val_loss": 0.28238230884075166,
  "val_acc1": 93.13999996337891,
  "val_time_s": 1.6815948630028288
}

                                                                                   

                                                    
{
  "epoch": 1,
  "lr": 0.0001996917333733128,
  "train_loss": 0.6306647899483684,
  "train_acc1": 94.72934472934473,
  "train_time_s": 29.54616035399522,
  "val_loss": 0.27480330369472505,
  "val_acc1": 93.37999986572265,
  "val_time_s": 0.9245453100011218
}

                                                                                   

                                                    
{
  "epoch": 2,
  "lr": 0.00019930684569549264,
  "train_loss"

stage	channels	min	max	mean	keep@0.5	rounded_keep@0.5	keep@0.7	rounded_keep@0.7	keep@0.8	rounded_keep@0.8	keep@0.9	rounded_keep@0.9	keep@0.95	rounded_keep@0.95
0	48	0.9691	0.9704	0.9698	48	48	48	48	48	48	48	48	48	48
1	80	0.9687	0.9706	0.9699	80	80	80	80	80	80	80	80	80	80
2	160	0.9685	0.9710	0.9699	160	160	160	160	160	160	160	160	160	160
3	224	0.9272	0.9720	0.9564	224	224	224	224	224	224	224	224	160	160

[exit=0] elapsed=4.3s

$ /usr/bin/python3 tools/export_pruned.py --config configs/ablations/hbcc_accuracy_small_pruning_mask.yaml --checkpoint runs_pruning_accuracy/hbcc_accuracy_small_pruning_mask/best.pth --output configs/generated_ablations/hbcc_accuracy_small_pruned_export.yaml --threshold 0.90 --min-channels 16 --round-to 8


{'experiment': {'name': 'hbcc_accuracy_small_pruned_export'}, 'data': {'name': 'cifar10', 'root': 'data', 'download': True, 'batch_size': 128, 'val_batch_size': 256, 'test_batch_size': 256, 'val_size': 5000, 'split_seed': 42, 'workers': 2, 'augment': True, 'randaugment': {'enabled': True, 'num_ops': 2, 'magnitude': 9}, 'random_erasing': {'p': 0.25, 'scale': [0.02, 0.2], 'ratio': [0.3, 3.3]}}, 'model': {'name': 'hbcc', 'num_classes': 10, 'use_coord': True, 'embed_dims': [48, 80, 160, 224], 'depths': [2, 2, 3, 1], 'mlp_ratios': 3.0, 'heads': [2, 2, 4, 4], 'head_dim': [16, 16, 16, 16], 'proposals': [[2, 2], [2, 2], [2, 2], [2, 2]], 'folds': [[4, 4], [2, 2], [1, 1], [1, 1]], 'similarities': ['cosine', 'cosine', 'cosine', 'cosine'], 'stage_modes': ['hybrid', 'hybrid', 'cluster', 'cluster'], 'local_branches': ['lbpconv', 'dwconv', 'identity', 'identity'], 'local_ratios': [0.5, 0.5, 0.0, 0.0], 'channel_shuffle': [True, True, False, False], 'norm': 'bn', 'stem_stride': 2, 'drop_path_rate': 0.0

```text
test:  90%|█████████ | 36/40 [00:01<00:00, 23.71it/s]
```


                                                                                   

                                                    
{
  "epoch": 0,
  "lr": 0.0001999229036240723,
  "train_loss": 1.9274168948502282,
  "train_acc1": 33.30662393162393,
  "train_time_s": 30.721542496001348,
  "val_loss": 1.4427494981765747,
  "val_acc1": 48.74,
  "val_time_s": 1.6862590960008674
}

                                                                                   

                                                    
{
  "epoch": 1,
  "lr": 0.0001996917333733128,
  "train_loss": 1.736692720668608,
  "train_acc1": 42.964298433048434,
  "train_time_s": 29.632129329002055,
  "val_loss": 1.2479526048660279,
  "val_acc1": 56.19999998779297,
  "val_time_s": 0.9216568830015603
}

                                                                                   

                                                   
{
  "epoch": 2,
  "lr": 0.00019930684569549264,
  "train_loss": 1.6210400582

## 8. Benchmark Matrix

Run this after training checkpoints exist. Missing checkpoints are skipped and the benchmark falls back to untrained weights only when no checkpoint is provided.

In [13]:
BENCHMARK_JOBS = [
    ("configs/baselines/resnet18_cifar.yaml", "runs_teacher/resnet18_cifar/best.pth"),
    ("configs/baselines/mobilenet_v2_cifar.yaml", "runs_baselines/mobilenet_v2_cifar/best.pth"),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml", "runs_baselines/shufflenet_v2_x1_0_cifar/best.pth"),
    ("configs/coc_cifar_baseline.yaml", COC_BASELINE_CHECKPOINT),
    ("configs/hbcc_current_reference.yaml", None),
    ("configs/hbcc_latency_tiny.yaml", "runs_students/hbcc_latency_tiny/best.pth"),
    ("configs/hbcc_latency_small.yaml", "runs_students/hbcc_latency_small/best.pth"),
    ("configs/hbcc_accuracy_small.yaml", "runs_accuracy/hbcc_accuracy_small/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml", "runs_accuracy/hbcc_accuracy_medium/best.pth"),
    ("configs/hbcc_latency_tiny.yaml", "runs_kd/hbcc_latency_tiny/best.pth"),
    ("configs/ablations/hbcc_tiny_hamming_late.yaml", "runs_ablations/hbcc_tiny_hamming_late/best.pth"),
    ("configs/ablations/hbcc_tiny_lbpconv.yaml", "runs_ablations/hbcc_tiny_lbpconv/best.pth"),
    (PRUNED_CONFIG, PRUNED_FINETUNE_CHECKPOINT),
]

if RUN_BENCHMARKS:
    for cfg, ckpt in BENCHMARK_JOBS:
        if not (ROOT / cfg).exists():
            print("Skip missing config:", cfg)
            continue
        benchmark(cfg, checkpoint=ckpt, debug=False, profile=True)


$ /usr/bin/python3 tools/benchmark.py --config configs/baselines/resnet18_cifar.yaml --output results/benchmark --batch-sizes 1 16 64 128 --warmup 30 --runs 100 --checkpoint runs_teacher/resnet18_cifar/best.pth --profile


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "resnet18_cifar",
  "config_id": "resnet18_cifar",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "params_total": 11173962,
  "params_trainable": 11173962,
  "model_size_mb": 42.66205596923828,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 2.9578875099832658,
  "throughput_b1": 501.4238532054755,
  "latency_ms_b16": 5.51588490998256,
  "throughput_b16": 2935.787314860939,
  "latency_ms_b64": 15.121669410000322,
  "throughput_b64": 4241.872236267145,
  "latency_ms_b128": 28.442211229994427,
  "throughput_b128": 4494.044451561998,
  "peak_

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "mobilenet_v2_cifar",
  "config_id": "mobilenet_v2_cifar",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "params_total": 2236682,
  "params_trainable": 2236682,
  "model_size_mb": 8.662788391113281,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 4.746510229961132,
  "throughput_b1": 212.10742571849732,
  "latency_ms_b16": 4.759487509945757,
  "throughput_b16": 3496.4027839401356,
  "latency_ms_b64": 6.669045200032997,
  "throughput_b64": 9698.424445895515,
  "latency_ms_b128": 13.09192840999458,
  "throughput_b128": 9792.179016474282,
  

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "shufflenet_v2_x1_0_cifar",
  "config_id": "shufflenet_v2_x1_0_cifar",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "params_total": 1263854,
  "params_trainable": 1263854,
  "model_size_mb": 4.883369445800781,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 5.931955429987283,
  "throughput_b1": 171.3556685988161,
  "latency_ms_b16": 6.446198240009835,
  "throughput_b16": 2518.767894472073,
  "latency_ms_b64": 6.710368339990964,
  "throughput_b64": 9483.540519619257,
  "latency_ms_b128": 13.623875369958114,
  "throughput_b128": 9421.68350

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "coc_cifar_baseline",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "params_total": 2395814,
  "params_trainable": 2395814,
  "model_size_mb": 9.139305114746094,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 11.3060830100585,
  "throughput_b1": 88.00656766255142,
  "latency_ms_b16": 11.260478289987077,
  "throughput_b16": 1433.0761511940218,
  "latency_ms_b64": 11.555746879967046,
  "throughput_b64": 5507.499751706508,
  "latency_ms_b128": 13.865490120006143,
  "throughput_b128": 9295.464630018834,
  "peak_memory_

...

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_accuracy_medium",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "params_total": 2840862,
  "params_trainable": 2836254,
  "model_size_mb": 10.8741455078125,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 14.579177310006344,
  "throughput_b1": 69.62748649948053,
  "latency_ms_b16": 14.62817660998553,
  "throughput_b16": 1108.1288646901505,
  "latency_ms_b64": 14.747060600057011,
  "throughput_b64": 4270.701309228898,
  "latency_ms_b128": 21.145328380007413,
  "throughput_b128": 6070.727011677004,
  "peak_memor

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_latency_tiny",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "params_total": 447746,
  "params_trainable": 447746,
  "model_size_mb": 1.719390869140625,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 6.46419684002467,
  "throughput_b1": 154.51546897913647,
  "latency_ms_b16": 6.665352479976718,
  "throughput_b16": 2380.3670232006175,
  "latency_ms_b64": 6.685242640014621,
  "throughput_b64": 9294.378833503013,
  "latency_ms_b128": 6.7314482999790926,
  "throughput_b128": 19303.278271906383,
  "peak_memory_mb"

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_tiny_hamming_late",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "params_total": 447746,
  "params_trainable": 447746,
  "model_size_mb": 1.719390869140625,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 5120,
  "latency_ms_b1": 6.543420989983133,
  "throughput_b1": 146.52082341355427,
  "latency_ms_b16": 6.586862440017285,
  "throughput_b16": 2416.855879167929,
  "latency_ms_b64": 6.701888980023796,
  "throughput_b64": 9620.413608528157,
  "latency_ms_b128": 6.798791120017995,
  "throughput_b128": 18891.197375064083,
  "peak_mem

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_tiny_lbpconv",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "params_total": 463258,
  "params_trainable": 459226,
  "model_size_mb": 1.78155517578125,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 6.314958869988914,
  "throughput_b1": 155.1292060290927,
  "latency_ms_b16": 6.9470931599789765,
  "throughput_b16": 2346.3224183077155,
  "latency_ms_b64": 6.546593999955803,
  "throughput_b64": 9873.621364584005,
  "latency_ms_b128": 6.602184629955445,
  "throughput_b128": 19382.63808238415,
  "peak_memory_mb": 

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_accuracy_small_pruned_export",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "params_total": 1538618,
  "params_trainable": 1535162,
  "model_size_mb": 5.8943634033203125,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 11.826729370004614,
  "throughput_b1": 84.47537828791826,
  "latency_ms_b16": 12.973588349996135,
  "throughput_b16": 1330.0947775523514,
  "latency_ms_b64": 12.356797370011918,
  "throughput_b64": 5189.434723785631,
  "latency_ms_b128": 16.153917839983478,
  "throughput_b128": 7978.94882512058

## 9. Read Metrics and Build Summary Tables

In [14]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def collect_train_metrics() -> pd.DataFrame:
    rows = []
    for metrics_path in ROOT.glob("runs*/**/metrics.jsonl"):
        records = read_jsonl(metrics_path)
        val_records = [r for r in records if "val_acc1" in r]
        if not val_records:
            continue
        best = max(val_records, key=lambda r: r["val_acc1"])
        test_records = [r for r in records if "test_acc1" in r]
        test_record = next((r for r in reversed(test_records) if r.get("epoch") == best.get("epoch")), test_records[-1] if test_records else {})
        train_records = [r for r in records if "train_acc1" in r]
        last_train = train_records[-1] if train_records else {}
        rows.append({
            "run_dir": str(metrics_path.parent.relative_to(ROOT)),
            "experiment": metrics_path.parent.name,
            "best_epoch": best.get("epoch"),
            "best_val_acc1": best.get("val_acc1"),
            "best_val_loss": best.get("val_loss"),
            "test_acc1": test_record.get("test_acc1"),
            "test_loss": test_record.get("test_loss"),
            "last_epoch": last_train.get("epoch"),
            "last_train_acc1": last_train.get("train_acc1"),
        })
    return pd.DataFrame(rows).sort_values("best_val_acc1", ascending=False) if rows else pd.DataFrame()


train_df = collect_train_metrics()
train_df

,run_dir,experiment,best_epoch,best_val_acc1,best_val_loss,test_acc1,test_loss,last_epoch,last_train_acc1
1,runs_accuracy/hbcc_accuracy_medium,hbcc_accuracy_medium,276,94.78,0.373902,94.63,0.379557,299,64.995103
5,runs_teacher/resnet18_cifar,resnet18_cifar,188,94.72,0.278742,94.08,0.299027,199,99.995548
2,runs_pruning_accuracy/hbcc_accuracy_small_prun...,hbcc_accuracy_small_pruning_mask,74,94.40,0.255789,93.88,0.264292,79,96.990741
0,runs_accuracy/hbcc_accuracy_small,hbcc_accuracy_small,283,94.22,0.402152,93.91,0.408748,299,59.101229
4,runs_baselines/shufflenet_v2_x1_0_cifar,shufflenet_v2_x1_0_cifar,193,92.30,0.344218,92.08,0.360092,199,99.975516
3,runs_baselines/mobilenet_v2_cifar,mobilenet_v2_cifar,197,92.18,0.374289,91.40,0.403211,199,99.864227
6,runs_coc_baseline/coc_cifar_baseline,coc_cifar_baseline,180,89.24,0.451053,87.89,0.486109,199,99.777422
7,runs_pruned_accuracy/hbcc_accuracy_small_prune...,hbcc_accuracy_small_pruned_export,74,87.24,0.440571,86.48,0.452237,79,82.830306


In [15]:
def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / "results" / "benchmark").glob("*.json"):
        rec = json.loads(path.read_text(encoding="utf-8"))
        rec["benchmark_file"] = str(path.relative_to(ROOT))
        rows.append(rec)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


bench_df = collect_benchmarks()
cols = [
    "config_id",
    "model_name",
    "params_total",
    "macs",
    "bops",
    "latency_ms_b1",
    "throughput_b16",
    "throughput_b64",
    "throughput_b128",
    "peak_memory_mb",
]
bench_df[[c for c in cols if c in bench_df.columns]] if not bench_df.empty else bench_df

,config_id,model_name,params_total,macs,bops,latency_ms_b1,throughput_b16,throughput_b64,throughput_b128,peak_memory_mb
0,hbcc_accuracy_small_pruned_export,hbcc,1538618,None,0,11.826729,1330.094778,5189.434724,7978.948825,109.592285
1,coc_cifar_baseline,hbcc,2395814,None,0,11.306083,1433.076151,5507.499752,9295.464630,71.798828
2,hbcc_accuracy_small,hbcc,1538618,None,0,11.865600,1306.199172,5296.459703,7878.664955,109.592285
3,hbcc_latency_small,hbcc,902932,None,0,8.162669,1990.919976,7494.185320,14354.881955,54.133301
4,hbcc_current_reference,hbcc,870100,None,0,6.761377,2342.966558,7408.672936,7560.882247,173.993164
5,hbcc_latency_tiny,hbcc,447746,None,0,6.464197,2380.367023,9294.378834,19303.278272,44.390625
6,hbcc_accuracy_medium,hbcc,2840862,None,0,14.579177,1108.128865,4270.701309,6070.727012,145.565918
7,hbcc_tiny_hamming_late,hbcc,447746,None,5120,6.543421,2416.855879,9620.413609,18891.197375,44.390625
8,mobilenet_v2_cifar,mobilenet_v2_cifar,2236682,None,0,4.746510,3496.402784,9698.424446,9792.179016,123.365234
9,hbcc_tiny_lbpconv,hbcc,463258,None,0,6.314959,2346.322418,9873.621365,19382.638082,116.450195


## 10. Pareto Summary

This cell combines training accuracy with benchmark records when names match. Review the mapping before using it as a final report.

In [16]:
def attach_accuracy(bench: pd.DataFrame, train_metrics: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        return bench
    out = bench.copy()
    out["acc1"] = None
    if train_metrics.empty:
        return out
    acc_source = "test_acc1" if "test_acc1" in train_metrics.columns else "best_val_acc1"
    acc_map = dict(zip(train_metrics["experiment"], train_metrics[acc_source].fillna(train_metrics["best_val_acc1"])))
    for idx, row in out.iterrows():
        name = row.get("config_id")
        out.at[idx, "acc1"] = acc_map.get(name)
    return out


def pareto_frontier(df: pd.DataFrame) -> pd.DataFrame:
    required = ["acc1", "params_total", "latency_ms_b1", "peak_memory_mb"]
    if df.empty or any(c not in df.columns for c in required):
        return pd.DataFrame()
    valid = df.dropna(subset=required).copy()
    keep = []
    for i, row in valid.iterrows():
        dominated = False
        for j, other in valid.iterrows():
            if i == j:
                continue
            better_or_equal = (
                other["acc1"] >= row["acc1"]
                and other["params_total"] <= row["params_total"]
                and other["latency_ms_b1"] <= row["latency_ms_b1"]
                and other["peak_memory_mb"] <= row["peak_memory_mb"]
            )
            strictly_better = (
                other["acc1"] > row["acc1"]
                or other["params_total"] < row["params_total"]
                or other["latency_ms_b1"] < row["latency_ms_b1"]
                or other["peak_memory_mb"] < row["peak_memory_mb"]
            )
            if better_or_equal and strictly_better:
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return valid.loc[keep].sort_values(["acc1", "latency_ms_b1"], ascending=[False, True])


combined_df = attach_accuracy(bench_df, train_df)
frontier_df = pareto_frontier(combined_df)

out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
if not combined_df.empty:
    combined_df.to_csv(out_dir / "combined_training_benchmark.csv", index=False)
if not frontier_df.empty:
    frontier_df.to_csv(out_dir / "pareto_frontier.csv", index=False)

frontier_df[[c for c in ["config_id", "acc1", "params_total", "macs", "bops", "latency_ms_b1", "peak_memory_mb"] if c in frontier_df.columns]]

,config_id,acc1,params_total,macs,bops,latency_ms_b1,peak_memory_mb
6,hbcc_accuracy_medium,94.63,2840862,None,0,14.579177,145.565918
10,resnet18_cifar,94.08,11173962,None,0,2.957888,326.364746
2,hbcc_accuracy_small,93.91,1538618,None,0,11.865600,109.592285
11,shufflenet_v2_x1_0_cifar,92.08,1263854,None,0,5.931955,94.832520
8,mobilenet_v2_cifar,91.4,2236682,None,0,4.746510,123.365234
1,coc_cifar_baseline,87.89,2395814,None,0,11.306083,71.798828


## 11. Published Results

Final headline table for reporting: Top-1 accuracy alongside params, MACs, BOPs, batch-1 latency, and peak memory for every model that has both a training run and a benchmark record. Sorted by accuracy, descending. This is the table to cite in the paper.


In [ ]:
def build_publication_table(df: pd.DataFrame) -> pd.DataFrame:
    """Rename/round combined training+benchmark records into a paper-ready table."""
    if df.empty:
        return df
    rename = {
        "config_id": "Model",
        "acc1": "Top-1 Acc (%)",
        "params_total": "Params",
        "macs": "MACs",
        "bops": "BOPs",
        "latency_ms_b1": "Latency b1 (ms)",
        "peak_memory_mb": "Peak Memory (MB)",
    }
    cols = [c for c in rename if c in df.columns]
    table = df[cols].rename(columns=rename).copy()
    if "Top-1 Acc (%)" in table.columns:
        table = table.sort_values("Top-1 Acc (%)", ascending=False)
    return table.reset_index(drop=True)


publication_df = build_publication_table(combined_df)
publication_df.to_csv(out_dir / "cifar10_published_results.csv", index=False)
publication_df


In [17]:
if RUN_PARETO_REPORT:
    run_py(["tools/pareto_report.py", "results/benchmark", "--output", "results/pareto.md"])


$ /usr/bin/python3 tools/pareto_report.py results/benchmark --output results/pareto.md


results/pareto.md

[exit=0] elapsed=0.6s


## Recommended Execution Order

1. Run environment checks and smoke.
2. Enable `RUN_TEACHER` and train ResNet-18.
3. Enable `RUN_COC_BASELINE` and train the CoC CIFAR baseline.
4. Enable `RUN_STUDENT_CE` and train HBCC Tiny/Small plus HBCC-Accuracy Small/Medium.
5. Enable `RUN_PROXY_SEARCH`; use the table to choose top candidates.
6. Enable `RUN_ABLATIONS` for Hamming/LBPConv.
7. Enable `RUN_KD` after the teacher checkpoint exists.
8. Enable `RUN_PRUNING` after `runs_accuracy/hbcc_accuracy_small/best.pth` exists; this prunes `hbcc_accuracy_small`.
9. Enable `RUN_BENCHMARKS` and build final tables.

Do not treat proxy accuracy or untrained benchmark records as final scientific evidence.